In [20]:
import os
import xml.etree.ElementTree as ET
from IPython.display import display, Markdown

import requests
from openai import OpenAI

In [22]:
PUBMED_SEARCH_URL = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
)
PUBMED_FETCH_URL = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
)

# Replace this with your email.
NCBI_EMAIL = "wale@hackbio.com"

# The model can be changed without editing the program.
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")

client = OpenAI()

In [23]:
def create_pubmed_query(question):
    """Ask the model to convert a question into a PubMed query."""

    response = client.responses.create(
        model=MODEL,
        instructions=(
            "Convert the research question into a concise PubMed query. "
            "Use biological keywords, Boolean operators, and synonyms where "
            "helpful. Return only the query."
        ),
        input=question,
    )

    return response.output_text.strip()


In [24]:
my_test_query = create_pubmed_query("does folic acid correct iron deficiency")

In [25]:
my_test_query

'("Folic Acid"[MeSH Terms] OR folic acid OR folate) AND ("Iron-Deficiency"[MeSH Terms] OR iron deficiency OR iron-deficiency anemia) AND (correct* OR treat* OR improv* OR effect*)'

In [26]:
def search_pubmed(query, maximum_results=5):
    """Search PubMed and return matching PubMed IDs."""

    search_query = f"({query}) AND hasabstract[Filter]"

    parameters = {
        "db": "pubmed",
        "term": search_query,
        "retmax": maximum_results,
        "retmode": "json",
        "sort": "relevance",
        "tool": "hackbio_pubmed_course",
        "email": NCBI_EMAIL,
    }

    response = requests.get(
        PUBMED_SEARCH_URL,
        params=parameters,
        timeout=30,
    )
    response.raise_for_status()

    return response.json()["esearchresult"]["idlist"]


In [27]:
my_test_pids = search_pubmed(my_test_query)

In [28]:
my_test_pids

['28034892', '39145520', '26198451', '36240826', '36263494']

In [8]:

def element_text(element):
    """Collect all text contained inside an XML element."""

    if element is None:
        return ""

    return " ".join("".join(element.itertext()).split())


In [9]:

def fetch_abstracts(pubmed_ids):
    """Retrieve article information for a list of PubMed IDs."""

    if not pubmed_ids:
        return []

    parameters = {
        "db": "pubmed",
        "id": ",".join(pubmed_ids),
        "retmode": "xml",
        "tool": "hackbio_pubmed_course",
        "email": NCBI_EMAIL,
    }

    response = requests.get(
        PUBMED_FETCH_URL,
        params=parameters,
        timeout=30,
    )
    response.raise_for_status()

    root = ET.fromstring(response.content)
    articles = []

    for record in root.findall(".//PubmedArticle"):
        citation = record.find("MedlineCitation")
        article = citation.find("Article")

        pmid = citation.findtext("PMID", default="")
        title = element_text(article.find("ArticleTitle"))

        abstract_parts = []
        for section in article.findall("./Abstract/AbstractText"):
            text = element_text(section)
            label = section.attrib.get("Label")

            if label and text:
                abstract_parts.append(f"{label}: {text}")
            elif text:
                abstract_parts.append(text)

        abstract = " ".join(abstract_parts)

        year = (
            article.findtext("./Journal/JournalIssue/PubDate/Year")
            or article.findtext("./ArticleDate/Year")
            or "Year not available"
        )

        if abstract:
            articles.append(
                {
                    "pmid": pmid,
                    "title": title,
                    "year": year,
                    "abstract": abstract,
                    "link": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
                }
            )

    return articles

In [30]:
my_test_articles = fetch_abstracts(my_test_pids)

In [31]:
my_test_articles

[{'pmid': '28034892',
  'title': 'How I treat anemia in pregnancy: iron, cobalamin, and folate.',
  'year': '2017',
  'abstract': 'Anemia of pregnancy, an important risk factor for fetal and maternal morbidity, is considered a global health problem, affecting almost 50% of pregnant women. In this article, diagnosis and management of iron, cobalamin, and folate deficiencies, the most frequent causes of anemia in pregnancy, are discussed. Three clinical cases are considered. Iron deficiency is the most common cause. Laboratory tests defining iron deficiency, the recognition of developmental delays and cognitive abnormalities in iron-deficient neonates, and literature addressing the efficacy and safety of IV iron in pregnancy are reviewed. An algorithm is proposed to help clinicians diagnose and treat iron deficiency, recommending oral iron in the first trimester and IV iron later. Association of folate deficiency with neural tube defects and impact of fortification programs are discussed

In [14]:

def review_abstracts(question, query, articles):
    """Ask the model to review the retrieved abstracts."""

    article_text = ""

    for number, article in enumerate(articles, start=1):
        article_text += f"""
            ARTICLE {number}
            PMID: {article["pmid"]}
            Title: {article["title"]}
            Year: {article["year"]}
            Link: {article["link"]}
            Abstract: {article["abstract"]}
        """

    instructions = """
        You are assisting a biologist with a small PubMed abstract review.
        
        Use only the supplied abstracts. Do not invent findings, methods, or
        references. If information is missing, say so.
        
        Prepare a report containing:
        
        1. Research question
        2. PubMed query
        3. A short summary of each article
        4. Common findings
        5. Important differences
        6. Limitations of the evidence
        7. References with PubMed IDs and links
        
        Place the relevant PMID after every article-level summary.
        Clearly state that the report is based only on abstracts.
    """

    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=f"""
            Research question: {question}
            PubMed query: {query}
            
            Retrieved articles:
            {article_text}
        """,
    )

    return response.output_text


In [32]:
my_test_rev = review_abstracts(
    question="does folic acid correct iron deficiency",
    query=my_test_query,
    articles=my_test_articles
    )

In [19]:
my_test_rev

'# Abstract-based report\n\n**Scope:** This report is based **only on the five supplied PubMed abstracts**. The abstracts do not provide enough information to determine whether folic acid alone corrects iron deficiency directly.\n\n## 1. Research question\n\n**Does folic acid, either alone or with iron, correct iron deficiency or iron-deficiency anemia?**\n\nBased on the supplied abstracts, the evidence primarily evaluates **iron supplementation** or **iron plus folic acid**, rather than folic acid alone.\n\n## 2. PubMed query\n\n> ("Folic Acid"[Mesh] OR folic acid[tiab] OR folate[tiab]) AND ("Iron Deficiency"[Mesh] OR "iron deficiency"[tiab] OR "iron-deficiency anemia"[tiab]) AND (treatment[tiab] OR correction[tiab] OR supplementation[tiab] OR response[tiab])\n\n## 3. Short summary of each article\n\n### Article 1: *Anemia in Infants and Children: Evaluation and Treatment* (2024)\n\nThis review states that nutritional iron deficiency is the most common cause of anemia in children. It 

In [33]:
display(Markdown(my_test_rev))

# Abstract-based report

**Scope:** This report is based **only on the supplied PubMed abstracts**. It does not assess the full texts, supplementary data, or individual trial reports.

## 1. Research question

**Does folic acid correct iron deficiency or iron-deficiency anemia?**

Based on the supplied abstracts, the evidence primarily evaluates **iron supplementation**, either alone or combined with folic acid. It does not directly establish whether **folic acid alone** corrects iron deficiency.

## 2. PubMed query

```text
("Folic Acid"[MeSH Terms] OR folic acid OR folate) AND
("Iron-Deficiency"[MeSH Terms] OR iron deficiency OR iron-deficiency anemia) AND
(correct* OR treat* OR improv* OR effect*)
```

## 3. Short summary of each article

### Article 1

This clinical review discusses anemia in pregnancy caused by iron, cobalamin, and folate deficiencies. It identifies iron deficiency as the most common cause and describes diagnostic approaches and treatment with oral or intravenous iron. Folate deficiency is discussed mainly in relation to neural tube defects and fortification, rather than as a treatment for iron deficiency. The abstract does not report evidence that folic acid alone corrects iron deficiency. **PMID: 28034892**  
[PubMed link](https://pubmed.ncbi.nlm.nih.gov/28034892/)

### Article 2

This 2024 Cochrane review included 57 trials involving 48,971 women. Iron supplementation alone probably reduced maternal iron deficiency and iron-deficiency anemia at term. Iron plus folic acid probably reduced maternal anemia and may have reduced maternal iron deficiency, but the iron-plus-folic-acid comparisons were based on relatively few trials and participants for several outcomes. Because the intervention included iron, these findings do not show that folic acid alone corrected iron deficiency. **PMID: 39145520**  
[PubMed link](https://pubmed.ncbi.nlm.nih.gov/39145520/)

### Article 3

This 2015 Cochrane review included 61 trials and found that preventive daily iron supplementation reduced maternal anemia, iron-deficiency anemia, and iron deficiency during pregnancy. It also evaluated iron plus folic acid, but the abstract’s principal results concern iron supplementation and does not provide evidence that folic acid alone reverses iron deficiency. Effects on other maternal and infant outcomes were less clear, with variation related to population risk and adherence. **PMID: 26198451**  
[PubMed link](https://pubmed.ncbi.nlm.nih.gov/26198451/)

### Article 4

This pooled analysis estimated the prevalence of iron, zinc, and folate deficiencies among preschool-aged children and non-pregnant women of reproductive age using data from 24 nationally representative surveys. It found a high global burden of micronutrient deficiency, including deficiencies involving iron and folate. It was an observational prevalence study, not a treatment study, and does not address whether folic acid corrects iron deficiency. **PMID: 36240826**  
[PubMed link](https://pubmed.ncbi.nlm.nih.gov/36240826/)

### Article 5

These pediatric recommendations distinguish iron-deficiency anemia from vitamin B12 or folic-acid-deficiency anemia. They recommend oral elemental iron for most children with iron-deficiency anemia, while folic acid is discussed as treatment for folic-acid deficiency and as part of preventive iron-folic-acid supplementation. The abstract does not state that folic acid alone treats iron deficiency; it recommends iron for iron-deficiency anemia. **PMID: 36263494**  
[PubMed link](https://pubmed.ncbi.nlm.nih.gov/36263494/)

## 4. Common findings

- **Iron supplementation is the intervention associated with correction or reduction of iron deficiency and iron-deficiency anemia.**
- The pregnancy reviews report reductions in maternal anemia, iron deficiency, and/or iron-deficiency anemia with iron supplementation.
- Iron plus folic acid was associated with reduced maternal anemia in the 2024 review and may reduce maternal iron deficiency, but the combination contains iron, so the independent effect of folic acid cannot be determined from these results.
- The clinical review and pediatric recommendations treat iron deficiency and folate deficiency as distinct nutritional problems requiring different diagnostic and therapeutic approaches.
- None of the supplied abstracts provides direct evidence that **folic acid monotherapy** corrects iron deficiency.

## 5. Important differences

- **Population:** Articles 1–3 focus mainly on pregnancy; Article 5 focuses on children; Article 4 concerns preschool-aged children and non-pregnant women.
- **Study type:** Articles 2 and 3 are systematic reviews of randomized or quasi-randomized trials. Article 1 is a clinical review with cases. Article 4 is a pooled analysis of population surveys. Article 5 is an expert guideline.
- **Intervention:** Articles 2 and 3 assess iron alone and iron combined with folic acid or other micronutrients. Article 5 recommends iron for iron-deficiency anemia and folic acid for folate-deficiency anemia. Article 4 assesses deficiency prevalence rather than treatment.
- **Certainty and sample size:** The 2024 review reports moderate- or low-certainty evidence for several outcomes, while some iron-plus-folic-acid outcomes are based on only one or a few trials and have low or very low certainty.
- **Outcome focus:** The reviews assess anemia and iron-status outcomes, whereas Article 4 estimates the population burden of deficiencies and Articles 1 and 5 provide clinical diagnostic and management guidance.

## 6. Limitations of the evidence

- **No direct test of the research question:** The supplied abstracts do not describe a comparison of folic acid alone versus placebo or no treatment for correction of iron deficiency.
- **Confounding by combined treatment:** Findings for iron plus folic acid cannot determine the specific contribution of folic acid because iron was administered simultaneously.
- **Indirect relevance:** Some articles concern anemia prevention during pregnancy rather than treatment of established iron deficiency.
- **Heterogeneity:** The Cochrane reviews included diverse populations, settings, interventions, adherence levels, and baseline risks. The 2015 review specifically notes heterogeneity and variation in results.
- **Limited certainty:** Several reported outcomes were graded low, very low, or uncertain certainty.
- **Different conditions:** Folate-deficiency anemia and iron-deficiency anemia are distinct conditions; evidence about folate deficiency should not be interpreted as evidence that folic acid replenishes iron stores.
- **Abstract-only assessment:** Details needed to evaluate dosing, duration, baseline iron status, diagnostic criteria, and subgroup effects are not fully available in the supplied abstracts.
- **Potential overlap:** Articles 2 and 3 are different editions of a Cochrane review and may include overlapping evidence, so their results should not necessarily be treated as independent studies.

## Overall conclusion

Based only on these abstracts, **folic acid alone has not been shown to correct iron deficiency**. The evidence supports **iron supplementation** as the treatment or preventive intervention associated with improved iron status and reduced iron-deficiency anemia. Iron-folic-acid combinations may improve outcomes, but the abstracts do not permit attribution of correction of iron deficiency to folic acid rather than to the iron component.

## 7. References

1. How I treat anemia in pregnancy: iron, cobalamin, and folate. 2017. **PMID: 28034892.**  
   https://pubmed.ncbi.nlm.nih.gov/28034892/

2. Daily oral iron supplementation during pregnancy. 2024. **PMID: 39145520.**  
   https://pubmed.ncbi.nlm.nih.gov/39145520/

3. Daily oral iron supplementation during pregnancy. 2015. **PMID: 26198451.**  
   https://pubmed.ncbi.nlm.nih.gov/26198451/

4. Micronutrient deficiencies among preschool-aged children and women of reproductive age worldwide: a pooled analysis of individual-level data from population-representative surveys. 2022. **PMID: 36240826.**  
   https://pubmed.ncbi.nlm.nih.gov/36240826/

5. Diagnosis, Treatment and Prevention of Nutritional Anemia in Children: Recommendations of the Joint Committee of Pediatric Hematology-Oncology Chapter and Pediatric and Adolescent Nutrition Society of the Indian Academy of Pediatrics. 2022. **PMID: 36263494.**  
   https://pubmed.ncbi.nlm.nih.gov/36263494/